# 🕸️ 03 - Echo-Chamber Graph & Network Centrality Mapping

### Uncovering Information Contagion & Amplification Loops

Sensational financial stories do not exist in a vacuum; they circulate in **echo chambers** where multiple outlets parrot identical buzzwords, anonymous sources, and tickers without independent verification.

In this notebook, we model cross-outlet media dynamics as an **attributed undirected network**:
- **Nodes**: Media outlets (attributed with average hype, category, and flagged ratio).
- **Edges**: Weighted narrative alignment combining Named Entity Jaccard similarity and top article lexical convergence.
- **Centrality**: PageRank and Degree Centrality identify the primary **echo epicenters**.
- **Modularity**: Greedy modularity community detection partitions the graph into distinct thematic echo chambers.

---

In [ ]:
import pandas as pd
import networkx as nx
from src.config import PROCESSED_DATA_DIR, FIGURES_DIR
from src.graph import EchoChamberGraphBuilder
from src.viz import plot_echo_chamber_network_plotly, export_pyvis_network_html

# Load enriched features
df = pd.read_parquet(PROCESSED_DATA_DIR / 'sample_features.parquet')

# Build Echo-Chamber Graph
builder = EchoChamberGraphBuilder()
G = builder.build_outlet_graph(df, edge_threshold=0.15)

print(f'Graph Nodes (Outlets): {len(G.nodes)}')
print(f'Graph Edges (Echo Links): {len(G.edges)}')

summary_data = []
for node, data in G.nodes(data=True):
    summary_data.append({
        'Outlet': node,
        'Category': data.get('category'),
        'Avg Hype': data.get('avg_hype_score'),
        'Community': data.get('community'),
        'PageRank': data.get('pagerank'),
        'Degree': data.get('degree_centrality')
    })

df_nodes = pd.DataFrame(summary_data).sort_values(by='PageRank', ascending=False)
df_nodes

## 2. Interactive Force-Directed Network Graph (Plotly)

- **Node Color**: Average BS / Hype score (Viridis/Plasma).
- **Node Size**: PageRank network influence (how central the outlet is to the broader echo system).
- **Hover**: Inspect community clusters, category, and flagged rates.

In [ ]:
fig_network = plot_echo_chamber_network_plotly(G)
fig_network.show()

## 3. Detailed Edge Analysis: Shared Narratives & Common Entities

Examine which outlets form the strongest narrative echo loops and which entities they co-amplify.

In [ ]:
edge_records = []
for u, v, d in G.edges(data=True):
    edge_records.append({
        'Source Outlet': u,
        'Target Outlet': v,
        'Echo Strength': d.get('weight'),
        'Co-Amplified Entities': ', '.join(d.get('common_entities', []))
    })

df_edges = pd.DataFrame(edge_records).sort_values(by='Echo Strength', ascending=False)
df_edges.head(10)

## 4. Export Physics-Based PyVis Simulation

Generate an interactive HTML graph using PyVis with real-time ForceAtlas2 physics.

In [ ]:
html_path = export_pyvis_network_html(G, FIGURES_DIR / 'echo_chamber_graph.html')
print(f'✓ PyVis graph exported to {html_path}')
print('You can open this HTML file directly in any browser for interactive physics manipulation!')
print('\nProceed to Notebook 04 for the live executive dashboard!')